# Kenya 2030 — PyPSA-Earth Results Analysis
Run cells in order. Update `NETWORK_PATH` in Cell 1 to point at your solved `.nc` file.

In [1]:
# ── Cell 1: Imports & load network ──────────────────────────────────────────
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import json, warnings
from importlib.metadata import version
from shapely.geometry import Point, LineString

print(version('pypsa'))

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ▶ Update this path to your solved network file
NETWORK_PATH = "results/2030_scenenario/networks/elec_s_15_ec_lcopt_CCL-Co2L0.5-3h.nc"

n = pypsa.Network(NETWORK_PATH)
print(f"Loaded: {NETWORK_PATH}")
print(f"  Buses      : {len(n.buses)}")
print(f"  Generators : {len(n.generators)}")
print(f"  Lines      : {len(n.lines)}")
print(f"  Snapshots  : {len(n.snapshots)}  ({n.snapshots[0]} → {n.snapshots[-1]})")

0.30.3


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\devsi\\OneDrive\\Documents\\IITB\\Research\\Energy System Modelling\\my model\\pypsa-earth\\results\\2030_scenenario\\networks\\elec_s_15_ec_lcopt_CCL-Co2L0.5-3h.nc'

---
## 1 · Existing vs optimised capacity per technology

In [ ]:
# ── Cell 2: Capacity table ───────────────────────────────────────────────────
gen = n.generators.copy()

# p_nom        = capacity before optimisation (existing floor)
# p_nom_opt    = capacity after optimisation
# new_build    = p_nom_opt - p_nom  (only meaningful for extendable generators)
gen['existing_MW']  = gen['p_nom']
gen['optimised_MW'] = gen['p_nom_opt'] if 'p_nom_opt' in gen.columns else gen['p_nom']
gen['new_build_MW'] = (gen['optimised_MW'] - gen['existing_MW']).clip(lower=0)

cap = gen.groupby('carrier')[['existing_MW', 'optimised_MW', 'new_build_MW']].sum()
cap = cap[cap['optimised_MW'] > 0].sort_values('optimised_MW', ascending=False)
cap['share_%'] = (cap['optimised_MW'] / cap['optimised_MW'].sum() * 100).round(1)

print("Capacity summary (MW)")
print("=" * 65)
print(f"{'Carrier':<20} {'Existing':>12} {'New build':>12} {'Optimised':>12} {'Share %':>8}")
print("-" * 65)
for carrier, row in cap.iterrows():
    print(f"{carrier:<20} {row.existing_MW:>12.1f} {row.new_build_MW:>12.1f} "
          f"{row.optimised_MW:>12.1f} {row['share_%']:>7.1f}%")
print("-" * 65)
print(f"{'TOTAL':<20} {cap.existing_MW.sum():>12.1f} {cap.new_build_MW.sum():>12.1f} "
      f"{cap.optimised_MW.sum():>12.1f}")

# Storage
if not n.storage_units.empty:
    su = n.storage_units.copy()
    su['optimised_MW'] = su['p_nom_opt'] if 'p_nom_opt' in su.columns else su['p_nom']
    su['energy_MWh']   = su['optimised_MW'] * su['max_hours']
    print("\nStorage units (MW / MWh)")
    print(su.groupby('carrier')[['optimised_MW','energy_MWh']].sum().round(1).to_string())

In [ ]:
# ── Cell 3: Capacity bar chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

x = np.arange(len(cap))
w = 0.35
ax.bar(x - w/2, cap['existing_MW'],  w, label='Existing',  color='#4C72B0')
ax.bar(x + w/2, cap['optimised_MW'], w, label='Optimised', color='#55A868')

ax.set_xticks(x)
ax.set_xticklabels(cap.index, rotation=30, ha='right')
ax.set_ylabel('Capacity (MW)')
ax.set_title('Kenya 2030 — existing vs optimised capacity')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('capacity_comparison.png', bbox_inches='tight')
plt.show()

---
## 2 · OSM cluster map — power mix at each bus

In [ ]:
# ── Cell 4: Cluster map with pie charts ─────────────────────────────────────
# Colour palette — extend if you have more carriers
COLORS = {
    'onwind'    : '#235ebc',
    'solar'     : '#f9d002',
    'geothermal': '#ba91b1',
    'hydro'     : '#08ad97',
    'ror'       : '#4adbc8',
    'OCGT'      : '#d35050',
    'oil'       : '#262626',
    'coal'      : '#707070',
    'battery'   : '#808080',
    'biomass'   : '#0c6013',
}
def carrier_color(c):
    return COLORS.get(c, '#aaaaaa')

buses = n.buses.copy()

# Use optimised capacity for the pie size
gen['cap'] = gen['optimised_MW']
bus_mix = gen.groupby(['bus', 'carrier'])['cap'].sum().unstack(fill_value=0)

# --- figure setup ---
fig, ax = plt.subplots(figsize=(9, 10))
ax.set_facecolor('#e8f4f8')

# Kenya bounding box
ax.set_xlim(33.8, 42.0)
ax.set_ylim(-5.0,  5.2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Kenya 2030 — cluster map (optimised capacity mix)', fontsize=12)
ax.grid(alpha=0.2)

# Draw transmission lines
for _, line in n.lines.iterrows():
    b0 = buses.loc[line.bus0]
    b1 = buses.loc[line.bus1]
    ax.plot([b0.x, b1.x], [b0.y, b1.y],
            color='black', lw=0.6, alpha=0.5, zorder=1)

# Draw pie chart at each bus
PIE_SCALE = 0.004   # degrees per MW^0.5 — tweak if pies overlap
for bus_id, row in buses.iterrows():
    if bus_id not in bus_mix.index:
        ax.plot(row.x, row.y, 'o', color='gray', ms=3, zorder=2)
        continue

    mix = bus_mix.loc[bus_id]
    mix = mix[mix > 0]
    total = mix.sum()
    if total == 0:
        ax.plot(row.x, row.y, 'o', color='gray', ms=3, zorder=2)
        continue

    radius = PIE_SCALE * np.sqrt(total)
    colors  = [carrier_color(c) for c in mix.index]

    # inset axes for the pie
    ax_pie = ax.inset_axes(
        [row.x - radius, row.y - radius, 2*radius, 2*radius],
        transform=ax.transData
    )
    ax_pie.pie(mix.values, colors=colors, startangle=90)
    ax_pie.set_zorder(3)

    ax.text(row.x, row.y + radius + 0.05, bus_id,
            ha='center', va='bottom', fontsize=6, zorder=5)

# Legend
all_carriers = gen['carrier'].unique()
handles = [mpatches.Patch(color=carrier_color(c), label=c) for c in all_carriers]
ax.legend(handles=handles, loc='lower left', fontsize=7,
          title='Technology', framealpha=0.9)

plt.tight_layout()
plt.savefig('cluster_map.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved cluster_map.png")

---
## 3 · Dispatch profile — 14-day window vs demand

In [ ]:
# ── Cell 5: Build dispatch & demand series ───────────────────────────────────
# Aggregated dispatch by carrier over all buses
dispatch = pd.DataFrame(index=n.snapshots)
for carrier in gen['carrier'].unique():
    gens_c = gen[gen['carrier'] == carrier].index
    # keep only generators that appear in p time-series
    cols = [g for g in gens_c if g in n.generators_t.p.columns]
    if cols:
        dispatch[carrier] = n.generators_t.p[cols].sum(axis=1)

# Add storage discharge (positive p)
if not n.storage_units_t.p.empty:
    for carrier in n.storage_units['carrier'].unique():
        su_c = n.storage_units[n.storage_units['carrier'] == carrier].index
        cols = [s for s in su_c if s in n.storage_units_t.p.columns]
        if cols:
            disch = n.storage_units_t.p[cols].clip(lower=0).sum(axis=1)
            dispatch[carrier + ' (discharge)'] = disch

# Total demand
demand = n.loads_t.p_set.sum(axis=1)

print(f"Dispatch carriers: {list(dispatch.columns)}")
print(f"Annual demand: {demand.sum()/1e6:.2f} TWh")

In [ ]:
# ── Cell 6: Plot 14-day dispatch ─────────────────────────────────────────────
# Pick a representative 14-day window (Jan by default — change as needed)
START = '2013-01-01'
END   = '2013-01-15'

d_slice  = dispatch.loc[START:END]
dm_slice = demand.loc[START:END]

# Carrier order: renewables first, then thermal
ORDER = ['onwind','solar','hydro','ror','geothermal',
         'biomass','OCGT','oil','coal']
cols = [c for c in ORDER if c in d_slice.columns]
cols += [c for c in d_slice.columns if c not in cols]   # any leftover
d_slice = d_slice[cols]

fig, ax = plt.subplots(figsize=(13, 5))

bottom = np.zeros(len(d_slice))
for carrier in d_slice.columns:
    vals = d_slice[carrier].values
    ax.fill_between(d_slice.index, bottom, bottom + vals,
                    label=carrier, color=carrier_color(carrier), alpha=0.85, step='post')
    bottom += vals

ax.plot(dm_slice.index, dm_slice.values,
        color='black', lw=1.8, ls='--', label='Demand', zorder=10)

ax.set_ylabel('Power (MW)')
ax.set_title(f'Kenya 2030 — dispatch vs demand  |  {START} to {END}')
ax.legend(loc='upper right', fontsize=8, ncol=2, framealpha=0.9)
ax.grid(axis='y', alpha=0.25)
ax.set_xlim(d_slice.index[0], d_slice.index[-1])

plt.tight_layout()
plt.savefig('dispatch_14day.png', bbox_inches='tight')
plt.show()
print("Saved dispatch_14day.png")

---
## 4 · Export GeoJSON for QGIS

In [ ]:
# ── Cell 7: Buses GeoJSON ────────────────────────────────────────────────────
def buses_to_geojson(n, path='buses.geojson'):
    """Export every bus with its optimised capacity mix as a GeoJSON point."""
    gen_opt = n.generators.copy()
    gen_opt['cap'] = gen_opt['p_nom_opt'] if 'p_nom_opt' in gen_opt.columns else gen_opt['p_nom']
    bus_cap = gen_opt.groupby(['bus','carrier'])['cap'].sum().unstack(fill_value=0)

    features = []
    for bus_id, row in n.buses.iterrows():
        props = {
            'bus_id'   : bus_id,
            'v_nom_kV' : row.get('v_nom', None),
            'country'  : row.get('country', 'KE'),
            'total_cap_MW': 0.0,
        }
        if bus_id in bus_cap.index:
            for carrier, mw in bus_cap.loc[bus_id].items():
                props[f'cap_{carrier}_MW'] = round(float(mw), 2)
            props['total_cap_MW'] = round(float(bus_cap.loc[bus_id].sum()), 2)

        # Average load at this bus
        loads_at_bus = n.loads[n.loads.bus == bus_id].index
        if loads_at_bus.any() and not n.loads_t.p_set.empty:
            lcols = [l for l in loads_at_bus if l in n.loads_t.p_set.columns]
            props['avg_load_MW'] = round(float(n.loads_t.p_set[lcols].sum(axis=1).mean()), 2) if lcols else 0.0

        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(row.x), float(row.y)]},
            'properties': props
        })

    fc = {'type': 'FeatureCollection', 'features': features}
    with open(path, 'w') as f:
        json.dump(fc, f, indent=2)
    print(f"Saved {len(features)} buses → {path}")

buses_to_geojson(n, 'buses.geojson')

In [ ]:
# ── Cell 8: Lines GeoJSON ────────────────────────────────────────────────────
def lines_to_geojson(n, path='lines.geojson'):
    """Export every transmission line with capacity & average flow."""
    avg_flow = None
    if hasattr(n.lines_t, 'p0') and not n.lines_t.p0.empty:
        avg_flow = n.lines_t.p0.mean()

    features = []
    for line_id, row in n.lines.iterrows():
        b0 = n.buses.loc[row.bus0]
        b1 = n.buses.loc[row.bus1]

        props = {
            'line_id'     : line_id,
            'bus0'        : row.bus0,
            'bus1'        : row.bus1,
            'v_nom_kV'    : float(row.get('v_nom', 0)),
            's_nom_MW'    : round(float(row.s_nom), 2),
            'length_km'   : round(float(row.get('length', 0)), 2),
        }
        if 's_nom_opt' in n.lines.columns:
            props['s_nom_opt_MW'] = round(float(row.s_nom_opt), 2)
        if avg_flow is not None and line_id in avg_flow.index:
            af = float(avg_flow[line_id])
            props['avg_flow_MW']    = round(af, 2)
            props['utilisation_pct'] = round(abs(af) / row.s_nom * 100, 1) if row.s_nom > 0 else 0.0

        features.append({
            'type': 'Feature',
            'geometry': {
                'type': 'LineString',
                'coordinates': [
                    [float(b0.x), float(b0.y)],
                    [float(b1.x), float(b1.y)]
                ]
            },
            'properties': props
        })

    fc = {'type': 'FeatureCollection', 'features': features}
    with open(path, 'w') as f:
        json.dump(fc, f, indent=2)
    print(f"Saved {len(features)} lines → {path}")

lines_to_geojson(n, 'lines.geojson')

In [ ]:
# ── Cell 9: (Optional) Generators GeoJSON — one point per generator ──────────
def generators_to_geojson(n, path='generators.geojson'):
    """Each generator as a point, with existing + optimised capacity."""
    features = []
    for gen_id, row in n.generators.iterrows():
        bus = n.buses.loc[row.bus]
        props = {
            'gen_id'        : gen_id,
            'carrier'       : row.carrier,
            'bus'           : row.bus,
            'existing_MW'   : round(float(row.p_nom), 2),
            'optimised_MW'  : round(float(row.p_nom_opt if 'p_nom_opt' in n.generators.columns else row.p_nom), 2),
            'extendable'    : bool(row.p_nom_extendable),
            'marginal_cost' : round(float(row.marginal_cost), 4),
        }
        props['new_build_MW'] = round(max(0.0, props['optimised_MW'] - props['existing_MW']), 2)

        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(bus.x), float(bus.y)]},
            'properties': props
        })

    fc = {'type': 'FeatureCollection', 'features': features}
    with open(path, 'w') as f:
        json.dump(fc, f, indent=2)
    print(f"Saved {len(features)} generators → {path}")

generators_to_geojson(n, 'generators.geojson')

print("\nAll GeoJSON files written. Load in QGIS:")
print("  buses.geojson      — cluster nodes + capacity mix + avg load")
print("  lines.geojson      — transmission lines + s_nom + avg flow + utilisation")
print("  generators.geojson — individual generators + existing/new capacity")